In [1]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from backbone import Backbone
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN

from pathlib import Path
from torch.utils.data import DataLoader, Subset

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
code = 'DualRetrieval'

dataset = EleHandler(subset='IDI_6')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=1024, shuffle=False)

In [3]:
MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)

MD_for_concepts.unfreeze()
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)
#concept_head.train(train_loader, test_loader, backbone = MD_for_concepts, num_epochs=50, predict_ele_SEEK = True)
#concept_head.test(test_loader, backbone = MD_for_concepts, predict_ele_SEEK = True)

r = Retrieval(experiment_code=code)
#r.evaluate_model(concept_head, ba= MD_for_concepts, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
r.evaluate_model(MD_finedtuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
evaluating backbone
Train Recalls:
Recall@1: 53.15%
Recall@5: 75.01%
Recall@10: 82.89%
Recall@20: 89.10%
Recall@100: 97.83%
Test Recalls:
Recall@1: 38.39%
Recall@5: 54.23%
Recall@10: 60.40%
Recall@20: 66.57%
Recall@100: 83.70%


{1: 0.38385376999238385,
 5: 0.5422696115765423,
 10: 0.6039603960396039,
 20: 0.6656511805026657,
 100: 0.837014470677837}

## Training with soft concepts corrected 50% of the time 

In [ ]:
import torch
def correct_or_soft(concept_logits, subject_SEEK, elephant_SEEK):
        # Randomly choose between oracle correction and perfect correction
        if torch.rand(1) < 0.5:
            return elephant_SEEK
        else:
            return concept_logits

def correct_or_hard(concept_logits, subject_SEEK, elephant_SEEK):
        # Randomly choose between oracle correction and perfect correction
        if torch.rand(1) < 0.5:
            return elephant_SEEK
        else:
            return SEEK.closest_valid_one_hot(concept_logits)

def hard(concept_logits, subject_SEEK, elephant_SEEK):
    return SEEK.closest_valid_one_hot(concept_logits)

In [ ]:
# Initialize Projector
pr = Projector(lr=1e-3, scale=64, margin=0.5, experiment_code='soft_corrected', intervention_fn=SEEK.oracle_correction)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=50
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": correct_or_soft,
    "0% Correction": None,
    "0% Correction Hard": hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

wandb: Currently logged in as: antoinesalaun (antoinesalaun-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:388: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Epoch 1/50 | Loss: 7.1801 | Batch R@1: 59.16% | Train: R@1=49.9% R@5=70.3% R@20=86.3% R@100=95.9% | Val: R@1=34.2% R@5=47.4% R@10=54.0% R@20=60.2% R@100=79.7%
Epoch 2/50 | Loss: 7.0339 | Batch R@1: 59.16% | Train: R@1=46.9% R@5=66.7% R@20=82.2% R@100=94.6% | Val: R@1=31.6% R@5=43.9% R@10=50.9% R@20=57.2% R@100=76.2%
Epoch 3/50 | Loss: 6.9370 | Batch R@1: 59.48% | Train: R@1=43.4% R@5=61.1% R@20=77.7% R@100=93.1% | Val: R@1=28.7% R@5=40.1% R@10=46.0% R@20=53.3% R@100=73.8%
Epoch 4/50 | Loss: 6.8427 | Batch R@1: 59.42% | Train: R@1=40.0% R@5=55.6% R@20=73.0% R@100=90.6% | Val: R@1=25.7% R@5=36.5% R@10=41.6% R@20=49.0% R@100=71.2%
Epoch 5/50 | Loss: 6.7450 | Batch R@1: 59.28% | Train: R@1=36.7% R@5=51.9% R@20=69.8% R@100=88.8% | Val: R@1=23.5% R@5=33.2% R@10=38.2% R@20=45.5% R@100=68.1%
Epoch 6/50 | Loss: 6.6439 | Batch R@1: 58.82% | Train: R@1=35.3% R@5=49.8% R@20=67.7%

## Training with hard concepts corrected 50% of the time 

In [ ]:
import torch


# Initialize Projector
pr = Projector(lr=5e-4, scale=64, margin=0.5, experiment_code='soft_corrected', intervention_fn=correct_or_hard)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": correct_or_soft,
    "50% Correction + hard": correct_or_hard,
    "0% Correction": None,
    "0% Correction Hard": hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

## Training with 0% corrected soft concepts

In [ ]:

# Initialize Projector
pr = Projector(lr=5e-4, scale=64, margin=0.5, experiment_code='soft_corrected', intervention_fn=None)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": correct_or_soft,
    "50% Correction + hard": correct_or_hard,
    "0% Correction": None,
    "0% Correction Hard": hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

## A tester :
- Soft concepts or correct 1/2
- Aggregated on Subject SEEK
- Just ELE_SEEK
- Corrected ELE_SEEK, test with diffferent level of 